# 03 — OpenAI Guarded Explanation Workflow

## Purpose

This notebook demonstrates controlled OpenAI integration for analyst-facing claim-triage explanations.

The LLM does not determine eligibility, coverage, payment, fraud, settlement, or customer-facing denial. It receives a minimized deterministic decision package and may generate only three non-authoritative explanation fields:

```text
case_summary
reviewer_note
next_step_message
```

## Protected workflow

```text
deterministic_triage
→ OpenAI structured explanation proposal
→ agent_content_safety_guardrail
→ response_guardrail
→ final protected response
```

## Safety boundaries

* Deterministic triage remains authoritative for outcome, triggering rule, precedence stage, decision reason, and rule trace.
* The OpenAI component cannot approve, deny, pay, settle, or determine fraud.
* Unsafe explanation wording is replaced with a deterministic fallback.
* Any attempted change to authoritative decision fields is blocked and recorded.
* The workflow remains decision support only; authorised reviewers retain responsibility for approved review, escalation, and exception-handling procedures.

## Scope of this notebook

This notebook validates:

1. Local environment and OpenAI configuration
2. Structured explanation generation
3. Semantic safety screening of LLM-generated content
4. Guarded LangGraph execution using a real OpenAI proposal builder
5. Comparison of safe and fallback behaviour across selected development claims

Held-out evaluation data is not loaded or used in this notebook.


## Step 1 — Environment, Runtime Data, and API Configuration

This step confirms that the notebook is running from the project structure, loads only approved runtime data, and verifies that the OpenAI API key is available through the local `.env` file.

The API key is never displayed, stored in the notebook output, or committed to GitHub.


In [1]:
# ============================================================
# Step 1: Environment, Runtime Data, and API Configuration
# ============================================================

from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv


CURRENT_DIRECTORY = Path.cwd().resolve()

if (CURRENT_DIRECTORY / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY
elif (CURRENT_DIRECTORY.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

dotenv_path = PROJECT_ROOT / ".env"

if not dotenv_path.exists():
    raise FileNotFoundError(
        "Local .env file was not found in the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not available in the environment."
    )

openai_model = os.getenv("OPENAI_EXPLANATION_MODEL")

if not openai_model:
    raise EnvironmentError(
        "OPENAI_EXPLANATION_MODEL is not available in the environment."
    )

from src.data_loader import load_runtime_data

data = load_runtime_data()

development_claims = data["development_claims"]

assert len(development_claims) == 165

print("Project root located:", PROJECT_ROOT.name)
print("Runtime data loaded successfully.")
print("Development claims available:", len(development_claims))
print("Configured explanation model:", openai_model)
print("API key available: True")

Project root located: DP_claims_triage
Runtime data loaded successfully.
Development claims available: 165
Configured explanation model: gpt-5.4-mini
API key available: True


## Step 2 — Live Structured Explanation and Semantic Safety Screening

This step generates a live OpenAI explanation for one development claim using only the deterministic decision package.

The generated explanation is then screened by the deterministic agent-content safety guardrail. The guardrail may retain safe content or replace unsafe wording with a deterministic fallback. In either case, the deterministic triage decision remains unchanged.

In [2]:
# ============================================================
# Step 2: Live Structured Explanation and Safety Screening
# ============================================================

from pprint import pprint

from src.agent.agent_content_guardrail import (
    apply_agent_content_safety_guardrail,
)
from src.agent.openai_explainer import (
    ALLOWED_EXPLANATION_FIELDS,
    build_openai_explanation_proposal,
)
from src.tools.deterministic_triage import run_deterministic_triage


claim_id = "CLM-000204"

tool_result = run_deterministic_triage(
    data=data,
    claim_id=claim_id,
)

openai_proposal = build_openai_explanation_proposal(
    tool_result=tool_result,
)

content_safety_result = apply_agent_content_safety_guardrail(
    tool_result=tool_result,
    proposed_content=openai_proposal,
)

print("Authoritative deterministic decision:")
pprint(tool_result["deterministic_decision"])

print("\nOpenAI structured explanation proposal:")
pprint(openai_proposal)

print("\nSemantic content-safety result:")
pprint(content_safety_result)

assert set(openai_proposal) == set(ALLOWED_EXPLANATION_FIELDS)

assert (
    tool_result["deterministic_decision"]["triage_outcome"]
    == "NOT_ELIGIBLE"
)

assert (
    tool_result["deterministic_decision"]["triggering_rule_id"]
    == "LIM-001"
)

assert content_safety_result["content_safety_status"] in {
    "SAFE",
    "FALLBACK_APPLIED",
}

assert set(content_safety_result["agent_content"]) == {
    "case_summary",
    "reviewer_note",
    "next_step_message",
}

print("\nValidation passed:")
print(
    "The model returned only permitted explanation fields, and the "
    "semantic guardrail produced safe analyst-facing content."
)

Authoritative deterministic decision:
{'claim_id': 'CLM-000204',
 'decision_reason': "The plan's annual claim allowance is exhausted.",
 'decision_support_only': True,
 'precedence_stage': 2,
 'rule_trace': [{'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'UNIQUE_MATCH',
                 'precedence_stage': 1,
                 'rule_id': 'DATA-001'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'UNIQUE_MATCH',
                 'precedence_stage': 1,
                 'rule_id': 'DATA-002'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': 'ACTIVE_ON_INCIDENT_DATE',
                 'precedence_stage': 1,
                 'rule_id': 'ELG-002'},
                {'evaluation': 'NOT_TRIGGERED',
                 'observed_value': {'coverage_lookup_status': 'UNIQUE_COVERAGE_RECORD',
                                    'plan_configuration_status': 'ACTIVE_CONFIGURATION_AVAILABLE'},
                 'prec

## Step 2B — Compact Limitation Summaries

The first live explanation was safe but its reviewer note became incomplete while repeating detailed system limitations.

This refinement sends concise, deterministic limitation summaries to the LLM while retaining the full limitations in the authoritative decision record. The objective is complete, readable analyst guidance without weakening auditability.

In [3]:
# ============================================================
# Step 2B: Compact Limitation Summary Validation
# ============================================================

import importlib

import src.agent.openai_explainer as openai_explainer

importlib.reload(openai_explainer)

build_openai_explanation_proposal = (
    openai_explainer.build_openai_explanation_proposal
)

refined_openai_proposal = build_openai_explanation_proposal(
    tool_result=tool_result,
)

refined_content_safety_result = apply_agent_content_safety_guardrail(
    tool_result=tool_result,
    proposed_content=refined_openai_proposal,
)

print("Refined OpenAI explanation proposal:")
pprint(refined_openai_proposal)

print("\nRefined semantic content-safety result:")
pprint(refined_content_safety_result)

assert refined_content_safety_result["content_safety_status"] in {
    "SAFE",
    "FALLBACK_APPLIED",
}

assert all(
    value.strip().endswith((".", "!", "?"))
    for value in refined_content_safety_result["agent_content"].values()
)

print(
    "\nValidation passed: analyst-facing content is complete and has passed "
    "semantic safety screening."
)

Refined OpenAI explanation proposal:
{'case_summary': 'Claim CLM-000204 routed to NOT_ELIGIBLE at precedence stage '
                 '2 because LIM-001 triggered: the annual claims used equals '
                 'the annual claim limit (2.0 of 2.0), indicating the plan’s '
                 'annual claim allowance is exhausted. Earlier checks in the '
                 'rule trace did not trigger.',
 'next_step_message': 'Proceed with the standard NOT_ELIGIBLE workflow and any '
                      'authorised human review or escalation steps applicable '
                      'to LIM-001 and the recorded system limitations.',
 'reviewer_note': 'This is a system triage recommendation for decision-support '
                  'only. Authorised reviewers may apply approved review, '
                  'escalation, or exception-handling procedures. Refer to the '
                  'authoritative decision record for full details, including '
                  'system limitations DEV-003, EX

## Step 3 — Live Guarded LangGraph Workflow

This step runs the real OpenAI structured explainer inside the protected LangGraph workflow.

The OpenAI output remains non-authoritative. It is screened for semantic safety and then passed through the authority guardrail before the final response is returned.

In [4]:
# ============================================================
# Step 3: Live OpenAI Guarded LangGraph Workflow
# ============================================================

from src.agent.langgraph_orchestrator import (
    run_langgraph_guarded_claim_triage,
)

live_langgraph_result = run_langgraph_guarded_claim_triage(
    data=data,
    claim_id="CLM-000204",
    proposal_builder=build_openai_explanation_proposal,
)

print("Workflow trace:")
pprint(live_langgraph_result["workflow_trace"])

print("\nContent safety result:")
pprint(live_langgraph_result["content_safety"])

print("\nAuthority guardrail:")
pprint(
    live_langgraph_result["final_response"]["authority_guardrail"]
)

print("\nFinal protected response:")
pprint(live_langgraph_result["final_response"])

assert live_langgraph_result["proposal_source"] == "CUSTOM"

assert (
    live_langgraph_result["content_safety"]["content_safety_status"]
    in {"SAFE", "FALLBACK_APPLIED"}
)

assert (
    live_langgraph_result["final_response"]["triage_outcome"]
    == "NOT_ELIGIBLE"
)

assert (
    live_langgraph_result["final_response"]["triggering_rule_id"]
    == "LIM-001"
)

assert [
    item["node"]
    for item in live_langgraph_result["workflow_trace"]
] == [
    "deterministic_triage",
    "explanation_proposal",
    "agent_content_safety_guardrail",
    "response_guardrail",
]

print(
    "\nValidation passed: real OpenAI explanation completed within the "
    "guarded LangGraph workflow without changing deterministic routing."
)

Workflow trace:
[{'authoritative': True,
  'node': 'deterministic_triage',
  'status': 'COMPLETED',
  'tool_version': 'rules_baseline_v1'},
 {'node': 'explanation_proposal',
  'proposal_source': 'CUSTOM',
  'proposed_field_names': ['case_summary',
                           'next_step_message',
                           'reviewer_note'],
  'status': 'COMPLETED'},
 {'content_safety_status': 'SAFE',
  'content_safety_violations': [],
  'fallback_used': False,
  'node': 'agent_content_safety_guardrail',
  'status': 'COMPLETED'},
 {'guardrail_status': 'ALIGNED',
  'node': 'response_guardrail',
  'status': 'COMPLETED'}]

Content safety result:
{'agent_content': {'case_summary': 'Claim CLM-000204 routed to NOT_ELIGIBLE at '
                                   'precedence stage 2 because LIM-001 '
                                   'triggered: the annual claim allowance is '
                                   'exhausted. The rule trace shows limit '
                                   'checks 

## Step 5 — Representative Multi-Outcome Live Validation

This step selects one development claim for each deterministic triage outcome and runs each through the live OpenAI-enabled LangGraph workflow.

The purpose is to confirm that explanation generation, semantic safety screening, and authority protection work consistently across PROCEED, INFO_REQUIRED, MANUAL_REVIEW, and NOT_ELIGIBLE outcomes.

Held-out evaluation data is not loaded or used.

In [5]:
# ============================================================
# Step 5: Representative Multi-Outcome Live Validation
# ============================================================

from IPython.display import display

OUTCOME_ORDER = [
    "PROCEED",
    "INFO_REQUIRED",
    "MANUAL_REVIEW",
    "NOT_ELIGIBLE",
]

representative_claims: dict[str, str] = {}
representative_decisions: dict[str, dict] = {}

for claim_id in development_claims["claim_id"].astype(str):
    deterministic_result = run_deterministic_triage(
        data=data,
        claim_id=claim_id,
    )

    deterministic_decision = deterministic_result[
        "deterministic_decision"
    ]

    outcome = deterministic_decision["triage_outcome"]

    if outcome in OUTCOME_ORDER and outcome not in representative_claims:
        representative_claims[outcome] = claim_id
        representative_decisions[outcome] = deterministic_decision

    if len(representative_claims) == len(OUTCOME_ORDER):
        break

assert set(representative_claims) == set(OUTCOME_ORDER)

print("Selected development claims:")
pprint(representative_claims)

validation_records = []

for expected_outcome in OUTCOME_ORDER:
    claim_id = representative_claims[expected_outcome]
    deterministic_decision = representative_decisions[expected_outcome]

    workflow_result = run_langgraph_guarded_claim_triage(
        data=data,
        claim_id=claim_id,
        proposal_builder=build_openai_explanation_proposal,
    )

    final_response = workflow_result["final_response"]
    content_safety = workflow_result["content_safety"]
    authority_guardrail = final_response["authority_guardrail"]

    validation_records.append(
        {
            "claim_id": claim_id,
            "expected_outcome": expected_outcome,
            "expected_rule": deterministic_decision[
                "triggering_rule_id"
            ],
            "final_outcome": final_response["triage_outcome"],
            "final_rule": final_response["triggering_rule_id"],
            "content_safety_status": content_safety[
                "content_safety_status"
            ],
            "fallback_used": content_safety["fallback_used"],
            "authority_guardrail_status": authority_guardrail["status"],
            "case_summary": final_response["agent_content"][
                "case_summary"
            ],
        }
    )

validation_summary = pd.DataFrame(validation_records)

display(
    validation_summary[
        [
            "claim_id",
            "expected_outcome",
            "expected_rule",
            "final_outcome",
            "final_rule",
            "content_safety_status",
            "fallback_used",
            "authority_guardrail_status",
        ]
    ]
)

assert (
    validation_summary["expected_outcome"]
    == validation_summary["final_outcome"]
).all()

assert (
    validation_summary["expected_rule"]
    == validation_summary["final_rule"]
).all()

assert validation_summary["content_safety_status"].isin(
    {"SAFE", "FALLBACK_APPLIED"}
).all()

assert (
    validation_summary["authority_guardrail_status"]
    == "ALIGNED"
).all()

print(
    "\nValidation passed: all four deterministic outcomes were preserved "
    "through the live OpenAI-enabled guarded LangGraph workflow."
)

Selected development claims:
{'INFO_REQUIRED': 'CLM-000071',
 'MANUAL_REVIEW': 'CLM-000131',
 'NOT_ELIGIBLE': 'CLM-000154',
 'PROCEED': 'CLM-000001'}


,claim_id,expected_outcome,expected_rule,final_outcome,final_rule,content_safety_status,fallback_used,authority_guardrail_status
0,CLM-000001,PROCEED,OUT-001,PROCEED,OUT-001,FALLBACK_APPLIED,True,ALIGNED
1,CLM-000071,INFO_REQUIRED,ELG-001,INFO_REQUIRED,ELG-001,SAFE,False,ALIGNED
2,CLM-000131,MANUAL_REVIEW,DEV-002,MANUAL_REVIEW,DEV-002,SAFE,False,ALIGNED
3,CLM-000154,NOT_ELIGIBLE,COV-001,NOT_ELIGIBLE,COV-001,SAFE,False,ALIGNED



Validation passed: all four deterministic outcomes were preserved through the live OpenAI-enabled guarded LangGraph workflow.


## Step 5B — Diagnose a Safe Fallback

One representative `PROCEED` case triggered the deterministic explanation fallback.

This diagnostic inspects the raw OpenAI proposal and the semantic guardrail result. The purpose is to understand the wording issue before deciding whether the LLM prompt needs refinement. The deterministic decision remains unchanged regardless of the result.

In [6]:
# ============================================================
# Step 5B: Diagnose the PROCEED Fallback
# ============================================================

proceed_claim_id = representative_claims["PROCEED"]

proceed_diagnostic = run_langgraph_guarded_claim_triage(
    data=data,
    claim_id=proceed_claim_id,
    proposal_builder=build_openai_explanation_proposal,
)

print("Claim ID:", proceed_claim_id)

print("\nRaw OpenAI proposal:")
pprint(proceed_diagnostic["agent_proposal"])

print("\nSemantic content-safety result:")
pprint(proceed_diagnostic["content_safety"])

print("\nFinal authority guardrail:")
pprint(
    proceed_diagnostic["final_response"]["authority_guardrail"]
)

assert (
    proceed_diagnostic["final_response"]["triage_outcome"]
    == "PROCEED"
)

assert (
    proceed_diagnostic["final_response"]["triggering_rule_id"]
    == "OUT-001"
)

assert (
    proceed_diagnostic["final_response"]["authority_guardrail"]["status"]
    == "ALIGNED"
)

print(
    "\nDiagnostic completed: deterministic PROCEED routing remained protected."
)

Claim ID: CLM-000001

Raw OpenAI proposal:
{'case_summary': 'Claim CLM-000001 routed to PROCEED at precedence stage 6 '
                 'because no earlier applicable deterministic rule triggered. '
                 'The trace shows standard checks passed, including data '
                 'matching, eligibility, coverage, limits, claim type, device '
                 'match, risk, and evidence sufficiency. This is a routing '
                 'result only and not an approval, payout, final denial, or '
                 'fraud determination.',
 'next_step_message': 'Proceed with standard processing workflow and any '
                      'required authorised human review steps per operating '
                      'procedure.',
 'reviewer_note': 'This is a system triage recommendation. Authorised '
                  'reviewers may apply approved review, escalation, or '
                  'exception-handling procedures using the authoritative '
                  'decision record. Syst

## Step 5C — Local Recheck of the PROCEED Explanation

The earlier `PROCEED` explanation was safe, but version 1 of the semantic guardrail treated its phrase “not an approval … final denial” as unsafe.

This local recheck uses the captured OpenAI proposal with the refined negation-aware guardrail. No new API call is made.

In [7]:
# ============================================================
# Step 5C: Local Recheck of the PROCEED Explanation
# ============================================================

import importlib

import src.agent.agent_content_guardrail as content_guardrail

importlib.reload(content_guardrail)

apply_agent_content_safety_guardrail = (
    content_guardrail.apply_agent_content_safety_guardrail
)

proceed_tool_result = run_deterministic_triage(
    data=data,
    claim_id=proceed_claim_id,
)

captured_proceed_proposal = proceed_diagnostic["agent_proposal"]

proceed_recheck = apply_agent_content_safety_guardrail(
    tool_result=proceed_tool_result,
    proposed_content=captured_proceed_proposal,
)

print("Captured OpenAI proposal:")
pprint(captured_proceed_proposal)

print("\nRefined guardrail result:")
pprint(proceed_recheck)

assert proceed_recheck["content_safety_status"] == "SAFE"
assert proceed_recheck["fallback_used"] is False
assert proceed_recheck["content_safety_violations"] == []
assert proceed_recheck["agent_content"] == captured_proceed_proposal

print(
    "\nValidation passed: the negated final-decision disclaimer is now "
    "correctly retained as safe content."
)

Captured OpenAI proposal:
{'case_summary': 'Claim CLM-000001 routed to PROCEED at precedence stage 6 '
                 'because no earlier applicable deterministic rule triggered. '
                 'The trace shows standard checks passed, including data '
                 'matching, eligibility, coverage, limits, claim type, device '
                 'match, risk, and evidence sufficiency. This is a routing '
                 'result only and not an approval, payout, final denial, or '
                 'fraud determination.',
 'next_step_message': 'Proceed with standard processing workflow and any '
                      'required authorised human review steps per operating '
                      'procedure.',
 'reviewer_note': 'This is a system triage recommendation. Authorised '
                  'reviewers may apply approved review, escalation, or '
                  'exception-handling procedures using the authoritative '
                  'decision record. System limitations no

## Step 5D — Local End-to-End Recheck of the PROCEED Workflow

This step reruns the captured `PROCEED` explanation through the LangGraph workflow after reloading the refined semantic guardrail.

No new OpenAI API call is made. The objective is to confirm that the complete workflow retains the safe explanation and returns an aligned authoritative response.

In [8]:
# ============================================================
# Step 5D: Local LangGraph Recheck Using Captured PROCEED Output
# ============================================================

import importlib

import src.agent.langgraph_orchestrator as langgraph_orchestrator

importlib.reload(langgraph_orchestrator)

run_langgraph_guarded_claim_triage = (
    langgraph_orchestrator.run_langgraph_guarded_claim_triage
)


def captured_proceed_builder(tool_result):
    return captured_proceed_proposal


proceed_langgraph_recheck = run_langgraph_guarded_claim_triage(
    data=data,
    claim_id=proceed_claim_id,
    proposal_builder=captured_proceed_builder,
)

print("Workflow trace:")
pprint(proceed_langgraph_recheck["workflow_trace"])

print("\nContent safety result:")
pprint(proceed_langgraph_recheck["content_safety"])

print("\nAuthority guardrail:")
pprint(
    proceed_langgraph_recheck["final_response"]["authority_guardrail"]
)

assert (
    proceed_langgraph_recheck["content_safety"]["content_safety_status"]
    == "SAFE"
)

assert (
    proceed_langgraph_recheck["content_safety"]["fallback_used"]
    is False
)

assert (
    proceed_langgraph_recheck["final_response"]["authority_guardrail"]["status"]
    == "ALIGNED"
)

assert (
    proceed_langgraph_recheck["final_response"]["triage_outcome"]
    == "PROCEED"
)

assert (
    proceed_langgraph_recheck["final_response"]["triggering_rule_id"]
    == "OUT-001"
)

print(
    "\nValidation passed: the captured PROCEED explanation is retained safely "
    "through the complete LangGraph workflow."
)

Workflow trace:
[{'authoritative': True,
  'node': 'deterministic_triage',
  'status': 'COMPLETED',
  'tool_version': 'rules_baseline_v1'},
 {'node': 'explanation_proposal',
  'proposal_source': 'CUSTOM',
  'proposed_field_names': ['case_summary',
                           'next_step_message',
                           'reviewer_note'],
  'status': 'COMPLETED'},
 {'content_safety_status': 'SAFE',
  'content_safety_violations': [],
  'fallback_used': False,
  'node': 'agent_content_safety_guardrail',
  'status': 'COMPLETED'},
 {'guardrail_status': 'ALIGNED',
  'node': 'response_guardrail',
  'status': 'COMPLETED'}]

Content safety result:
{'agent_content': {'case_summary': 'Claim CLM-000001 routed to PROCEED at '
                                   'precedence stage 6 because no earlier '
                                   'applicable deterministic rule triggered. '
                                   'The trace shows standard checks passed, '
                                   'includi